In [1]:
#Are High-Paid Running Backs Worth It??? A NFL Cap Analysis (2011 - 2025)

#Introduciton
This project was made to analyze whether teams that invest heavily in running backs achieve postseason success. In recent years there has been a anti-running back movement on the internet when it comes to giving a running back high paying money. Using Salary cap data from Spotrac and team performance data from Pro Football Reference, I have tracked the top 5 highest-paid running backs by cpa hit and contract average each season from 2011 to 2025 and analyizd how those teams finished

SyntaxError: invalid syntax (3568992543.py, line 4)

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('database/rb_analysis.db')
print("Connected to database successfully!")

In [ ]:
#Data Overview
The dataset consists of two tables:
- **rb_top_5**: top paid running backs each season by contract average and/or cap hit
- **team_data**: season outcomes for those teams including wins, losses, and playoff result

Data spans from **2011 to 2025**: chosen because the 2011 CBA introduced the current 
rookie wage scale and standardized contract structures, making salary comparisons 
consistent across all seasons.

In [ ]:
#Preview rb_top_5 table
print("=== RB TOP 5 ===")
rb_df = pd.read_sql("SELECT * FROM rb_top_5 LIMIT 10", conn)
print(rb_df.to_string())

#Preview team_data table
print("\n=== TEAM DATA ===")
team_df = pd.read_sql("SELECT * FROM team_data LIMIT 10", conn)
print(team_df.to_string())

In [ ]:
#Analysis
Now we join both tables to examine how teams with a top-paid running back actually performed each season.

In [ ]:
# Join both tables to see RB salary vs team outcome
query = '''
    SELECT 
        r.player_name,
        r.team,
        r.season,
        r.contract_average,
        r.cap_hit,
        r.cap_percentage,
        t.wins,
        t.losses,
        t.made_playoffs,
        t.playoff_result
    FROM rb_top_5 r
    JOIN team_data t ON r.team = t.team AND r.season = t.season
    ORDER BY r.season DESC
'''
df = pd.read_sql(query, conn)
df

In [ ]:
#Key Findings

#Finding 1: Most teams with a top-paid RB missed the playoffs
Nearly 2 out of every 3 teams that committed a lot of money to a running back failed to 
even make the playoffs that season.

In [ ]:
#Playoff outcome distribution
query = '''
    SELECT 
        COALESCE(playoff_result, 'Missed Playoffs') as outcome,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM team_data), 2) as percentage
    FROM team_data
    GROUP BY outcome
    ORDER BY count DESC
'''
df_outcomes = pd.read_sql(query, conn)
df_outcomes

In [ ]:
#Chart 1: Playoff Outcome Distribution
img = plt.imread('charts/chart1_outcome_distribution.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#Finding 2: Only 1 out of 105 team seasons resulted in a Super Bowl win
The 2023 Philadelphia Eagles with Saquon Barkley are the only team in this dataset 
to win the Super Bowl while having a top-paid running back since 2011. Intertestingly, Barkley only 
qualified by contract average, and not cap hit. Meaning the Eagles were not even 
taking a significant cap burden from him that season. Which allowed the Eagles to structure the salary cap efficiently over three years.

In [ ]:
#Super Bowl wins by top paid RB teams
query = '''
    SELECT 
        COUNT(*) as total_team_seasons,
        SUM(CASE WHEN playoff_result = 'Super Bowl Win' THEN 1 ELSE 0 END) as super_bowl_wins,
        ROUND(SUM(CASE WHEN playoff_result = 'Super Bowl Win' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as win_percentage
    FROM team_data
'''
df_sb = pd.read_sql(query, conn)
df_sb

In [ ]:
#Chart 2: Average Wins by Playoff Result
img = plt.imread('charts/chart2_avg_wins.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#Finding 3: Paying more at RB does not correlate with more wins
Teams that missed the playoffs actually had a higher average cap hit at RB than 
teams that made the Super Bowl. This data tells us that investing heavily in a running 
back not only fails to guarantee success but may actually signal poor roster construction by the teams front office.

In [ ]:
#Average cap hit by playoff result
query = '''
    SELECT 
        COALESCE(t.playoff_result, 'Missed Playoffs') as outcome,
        ROUND(AVG(r.cap_hit), 2) as avg_cap_hit,
        ROUND(AVG(r.cap_percentage), 2) as avg_cap_percentage
    FROM rb_top_5 r
    JOIN team_data t ON r.team = t.team AND r.season = t.season
    GROUP BY outcome
    ORDER BY avg_cap_hit DESC
'''
df_cap = pd.read_sql(query, conn)
df_cap

In [ ]:
#Chart 3: Cap Hit vs Wins Scatter Plot
img = plt.imread('charts/chart3_caphit_vs_wins.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#Finding 4: The trend has not improved over time
Even as NFL teams have become more analytics driven, the pattern remains consistent, teams continue to overpay at running back and continue to underperform as a result. 
The data shows no meaningful improvement in win totals for high paid RB teams over the 2011-2025 window.

In [ ]:
#Chart 4: Pie Chart Outcome Breakdown
img = plt.imread('charts/chart4_pie_outcomes.png')
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

#Chart 5: Cap Percentage Over Time
img = plt.imread('charts/chart5_cap_pct_over_time.png')
plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
#Conclusion

The data tells us that while running backs can be valuable assests to a teams success, not only does having a premium running back not guarantee success, not even a playoff birth.

Key takeaways:
- **63.81%** of teams with a top-paid RB missed the playoffs entirely
- Only **0.95%** of team seasons resulted in a Super Bowl win
- Teams that missed the playoffs had a **higher** average RB cap hit than Super Bowl teams
- The one exception — the 2023 Philadelphia Eagles — only qualified by contract average, not cap hit

This analysis supports the widely held analytics propaganda that the running 
back position is the least valuable position relative to cost in the NFL. Teams that 
allocate significant cap space to a running back are likely sacrificing depth and 
talent at more impactful positions like quarterback, offensive line, defensive backs, skills players, etc.

In [ ]:
#Close database connection
conn.close()
print("Analysis complete!")